In [1]:
#reusamos los codigos de la clase
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

#con el grafo
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph, h_dict=None):
        super().__init__(initial, goal)
        self.graph = graph
        self.h_dict = h_dict or {}

    def actions(self, state):
        return list(self.graph[state].keys())

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    def h(self, state):
        return self.h_dict.get(state, 0)

#los nodos db
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        node, path_back = self, []
        while node:
            path_back.append(node.state)
            node = node.parent
        return path_back[::-1]

    def expand(self, problem):
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

#el mapa de la clase y h
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}
#h
straight_line_distance = {
    'Arad': 366, 'Bucarest': 0, 'Craiova': 160, 'Dobreta': 242,
    'Eforie': 161, 'Fagaras': 178, 'Giurgiu': 77, 'Hirsova': 151,
    'Iasi': 226, 'Lugoj': 244, 'Mehadia': 241, 'Neamt': 234,
    'Oradea': 380, 'Pitesti': 98, 'Rimnicu Vilcea': 193, 'Sibiu': 253,
    'Timisoara': 329, 'Urziceni': 80, 'Vaslui': 199, 'Zerind': 374,
}

In [6]:
#busuqeda hill climbing
def hill_climbing(problem):
    #corre inical
    current = Node(problem.initial)
    evaluaciones = 0

    print("Iniciando...")
    
    while True:
        evaluaciones += 1
        neighbors = current.expand(problem)
        
        if not neighbors:
            break

        #selecciona el menor vecino para la h
        best_neighbor = min(neighbors, key=lambda node: problem.h(node.state))

        print(f"Paso {evaluaciones}: Estado actual '{current.state}' (h={problem.h(current.state)}) "
              f"-> Mejor vecino '{best_neighbor.state}' (h={problem.h(best_neighbor.state)})")

        #si ya no mejora
        if problem.h(best_neighbor.state) >= problem.h(current.state):
            print("\n No hay vecinos con menor h(n).")
            break

        #si avanza
        current = best_neighbor

        #avisa q termina
        if problem.is_goal(current.state):
            print("\nEl metodo termina ")
            break

    return current, evaluaciones

In [7]:
#se demilita de donde a donde
prob = GraphProblem('Arad', 'Bucarest', romania, straight_line_distance)

#comienza el metodo por busqueda de Hill Climbing
nodo_final, total_pasos = hill_climbing(prob)

print(f"Ruta encontrada: {nodo_final.path()}")
print(f"g: {nodo_final.path_cost} km")
print(f"Pasos realizadas: {total_pasos}")

Iniciando...
Paso 1: Estado actual 'Arad' (h=366) -> Mejor vecino 'Sibiu' (h=253)
Paso 2: Estado actual 'Sibiu' (h=253) -> Mejor vecino 'Fagaras' (h=178)
Paso 3: Estado actual 'Fagaras' (h=178) -> Mejor vecino 'Bucarest' (h=0)

El metodo termina 
Ruta encontrada: ['Arad', 'Sibiu', 'Fagaras', 'Bucarest']
g: 450 km
Pasos realizadas: 3
